# 07 – Triangulare Membranelemente

Eine **Membran** ist eine dünne Struktur, die nur Zugspannung trägt (keine Biege- oder Druckspannung).
Beispiele sind Trampolinmatten, Trommelfelle oder gespannte Folien. Unter einer
**homogenen Vorspannung** $\sigma_0$ und einem **Querdruck** $p_0$ verformt sich die Membran
in Normalrichtung um die kleine Auslenkung $w(x,y)$.

In diesem Notebook berechnen wir $w(x,y)$ für eine rechteckige Membran mit
**linearen Dreieckselementen**. Pro Knoten gibt es **einen Freiheitsgrad**: die Auslenkung $w$
aus der Membranebene.

## Differentialgleichung

Aus dem Gleichgewicht am Membranelement folgt die Poisson-Gleichung

$$
\nabla^2 w(x,y) \;=\; -\frac{p_0}{\sigma_0\,t}
\qquad \text{im Gebiet } \Omega,
\qquad w = 0 \;\text{auf}\; \partial\Omega .
$$

$\sigma_0$ ist die Vorspannung [Pa], $t$ die Membrandicke [m], $p_0$ der Querdruck [Pa].

## Vergleich mit Stab/Balken

| | Stab/Balken (1D) | Membran (2D) |
|---|---|---|
| Geometrie | Linie | Dreiecksfläche |
| Freiheitsgrade pro Knoten | 2 oder 3 | **1** ($w$) |
| Lokale ESM | $4 \times 4$ bzw. $6 \times 6$ | $3 \times 3$ |
| Belastung | Knoten- oder Linienlast | Flächenlast $p_0$ |

## Lineares Dreieckselement

Ein Element hat drei Knoten $1, 2, 3$ mit Koordinaten $(x_i, y_i)$ und Auslenkungen $w_i$.
Innerhalb des Elements interpolieren wir $w$ linear

$$
w(x,y) \;=\; N_1(x,y)\,w_1 + N_2(x,y)\,w_2 + N_3(x,y)\,w_3 .
$$

## Formfunktionen

Wie in der Vorlesung hergeleitet folgen die linearen Formfunktionen der Form

$$
\boldsymbol{N}(x,y) \;=\; \frac{1}{2 \Omega^e}\,\bigl[\boldsymbol{b}_0^{\!\top} + \boldsymbol{b}_x^{\!\top}\,x + \boldsymbol{b}_y^{\!\top}\,y\bigr],
$$

wobei $\boldsymbol{N} = [N_1, N_2, N_3]$ die Formfunktionen pro Knoten sammelt.

mit der Elementfläche

$$
\Omega^e \;=\; \tfrac{1}{2}\bigl[(x_2-x_1)(y_3-y_1) - (y_2-y_1)(x_3-x_1)\bigr]
$$

mit den drei Koeffizienten-Vektoren (zyklische Vertauschung $1 \to 2 \to 3 \to 1$):

$$
\boldsymbol{b}_0 = \begin{pmatrix} x_2 y_3 - x_3 y_2 \\ x_3 y_1 - x_1 y_3 \\ x_1 y_2 - x_2 y_1 \end{pmatrix},
\qquad
\boldsymbol{b}_x = \begin{pmatrix} y_2 - y_3 \\ y_3 - y_1 \\ y_1 - y_2 \end{pmatrix},
\qquad
\boldsymbol{b}_y = \begin{pmatrix} x_3 - x_2 \\ x_1 - x_3 \\ x_2 - x_1 \end{pmatrix}.
$$

Da die $N_i$ **linear** in $x, y$ sind, sind ihre Ableitungen **konstant** je Element

$$
\frac{\partial \boldsymbol{N}}{\partial x} = \frac{\boldsymbol{b}_x^{\!\top}}{2 \Omega^e}, \qquad
\frac{\partial \boldsymbol{N}}{\partial y} = \frac{\boldsymbol{b}_y^{\!\top}}{2 \Omega^e} .
$$

## Elementsteifigkeitsmatrix und Lastvektor

Schreibt man $\sigma_0\,t$ in den Nenner des Lastterms, also $\bar{p} = p_0/(\sigma_0\,t)$, verschwindet der Vorfaktor in der Steifigkeitsmatrix

$$
\underline{\underline{K}}^e \;=\; \Omega^e \,\cdot\, \underline{\underline{B}}^{\!\top}\,\underline{\underline{B}},
\qquad
\boldsymbol{p}_e \;=\; \frac{\bar{p}\,\Omega^e}{3}\,
\begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix}
\;=\; \frac{p_0\,\Omega^e}{3\,\sigma_0\,t}\,
\begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix},
$$

mit der Gradientenmatrix

$$
\underline{\underline{B}} \;=\;
\frac{1}{2\Omega^e}
\begin{bmatrix} \boldsymbol{b}_x^{\!\top} \\ \boldsymbol{b}_y^{\!\top} \end{bmatrix}
\;=\;
\frac{1}{2\Omega^e}
\begin{pmatrix}
 y_2-y_3 & y_3-y_1 & y_1-y_2 \\
 x_3-x_2 & x_1-x_3 & x_2-x_1
\end{pmatrix}\;\;(2 \times 3) .
$$

Jeder Knoten erhält gemäss $\boldsymbol{p}_e$ ein Drittel der Flächenlast.

> **Nur in Google Colab nötig:** Die folgende Zelle klont das Repository. Lokal oder in JupyterLite kann sie übersprungen werden.

In [ ]:
import os, sys
if not os.path.exists("FEM"):
    !git clone --depth=1 -q https://github.com/Boscij/FEM.git
if "FEM/content" not in sys.path:
    sys.path.insert(0, "FEM/content")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Modelldefinition und Vernetzung

**Beispiel:** Rechteckige Membran $1\,\mathrm{m} \times 0.8\,\mathrm{m}$, Dicke $0.5\,\mathrm{mm}$,
vorgespannt mit $\sigma_0 = 50\,\mathrm{MPa}$, Druck $p_0 = 5\,\mathrm{kPa}$. Alle Ränder eingespannt ($w=0$).

Wir benutzen ein **regelmässiges Gitter** und teilen jede Gitterzelle in zwei Dreiecke.
Einheiten: alles in SI (m, Pa).

In [ ]:
# Membran-Geometrie und Parameter
Lx, Ly = 1.0, 0.8    # Membran-Abmessungen [m]
t      = 0.5e-3      # Dicke               [m]
sigma0 = 50e6        # Vorspannung         [Pa]
p0     = 5e3         # Querdruck           [Pa]

# Knoten auf einem regelmaessigen Gitter
nx, ny = 25, 21      # Anzahl Knoten in jeder Richtung
x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)
X, Y = np.meshgrid(x, y)
nodes = np.column_stack((X.ravel(), Y.ravel()))

# Jede Gitterzelle in 2 Dreiecke aufteilen (immer entgegen Uhrzeigersinn)
elements = []
for i in range(ny - 1):
    for j in range(nx - 1):
        sw = i*nx + j          # south-west
        se = sw + 1            # south-east
        nw = sw + nx           # north-west
        ne = sw + nx + 1       # north-east
        elements.append([sw, se, ne])    # unteres Dreieck
        elements.append([sw, ne, nw])    # oberes  Dreieck
elements = np.array(elements)

# Randknoten: alles was auf der Aussenkante des Rechtecks liegt
tol = 1e-9
is_boundary = ((nodes[:,0] < tol) | (nodes[:,0] > Lx - tol) |
               (nodes[:,1] < tol) | (nodes[:,1] > Ly - tol))

print(f"Knoten:    {len(nodes)}  ({is_boundary.sum()} davon am Rand)")
print(f"Elemente:  {len(elements)}")

In [ ]:
# Netz visualisieren
fig, ax = plt.subplots(figsize=(8, 7))
ax.set_aspect("equal")
ax.triplot(nodes[:,0], nodes[:,1], elements, color="k", linewidth=0.4)
ax.plot(nodes[~is_boundary,0], nodes[~is_boundary,1], "ko", markersize=3, label="Innenknoten")
ax.plot(nodes[ is_boundary,0], nodes[ is_boundary,1], "go", markersize=5, label="Randknoten (w=0)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title(f"Dreiecksvernetzung  ({len(elements)} Elemente)")
ax.legend()
plt.show()

## Elementsteifigkeitsmatrix

Wir implementieren die Formel von oben direkt. Die Funktion liefert $\underline{\underline{K}}^e$ ($3 \times 3$) und den geometrischen Anteil von $\boldsymbol{p}_e$ ($3$) **ohne** den Faktor $\bar{p} = p_0/(\sigma_0\,t)$, den wir später vor den globalen Lastvektor ziehen.

In [ ]:
def element_stiffness(xy_e):
    """K_e (3x3) und Last-Anteil p_e (3) fuer 3-Knoten-Membranelement.
    xy_e: (3,2) Knotenkoordinaten."""
    x1, y1 = xy_e[0]
    x2, y2 = xy_e[1]
    x3, y3 = xy_e[2]

    # Elementflaeche
    omega_e = ((x2 - x1)*(y3 - y1) - (y2 - y1)*(x3 - x1)) / 2

    # Gradientenmatrix B (2 x 3)
    B = np.array([[y2 - y3, y3 - y1, y1 - y2],
                  [x3 - x2, x1 - x3, x2 - x1]]) / (2*omega_e)

    K_e = omega_e * B.T @ B           # 3 x 3
    p_e = np.full(3, omega_e/3)       # gleichmaessige Flaechenlast
    return K_e, p_e


# Test am ersten Element
Ke, pe = element_stiffness(nodes[elements[0]])
print("Erstes Element:  Knoten", elements[0])
print(f"  Omega^e        = {pe.sum()*3:.6f} m^2")
print(f"  K_e (normiert) =\n{Ke}")
print(f"  p_e (normiert) = {pe}")

## Assemblierung der globalen Matrix

Jeder Element-Beitrag wird an den Knotenindizes des Elements in $\underline{\underline{K}}$
und $\boldsymbol{f}$ aufaddiert. Da pro Knoten **nur ein Freiheitsgrad** existiert ($w$),
sind die globalen DOF-Indizes einfach die Knotennummern.

In [ ]:
def assemble(nodes, elements):
    """Assembliert die normierte Steifigkeitsmatrix K und Lastvektor p."""
    n_dof = len(nodes)
    K = np.zeros((n_dof, n_dof))
    p = np.zeros(n_dof)
    for el in elements:
        Ke, pe = element_stiffness(nodes[el])
        K[np.ix_(el, el)] += Ke
        p[el]             += pe
    return K, p


K, p = assemble(nodes, elements)
print(f"K: {K.shape},   Rang = {np.linalg.matrix_rank(K)}  (= n-1 wegen Konstanten-Nullmode)")
print(f"Symmetrie  max|K - K^T| = {np.max(np.abs(K - K.T)):.2e}")

## Lösen mit Dirichlet-Randbedingungen

Am Rand gilt $w = 0$. Wir partitionieren in freie ("F") und gesperrte Knoten und lösen
nur das reduzierte System

$$
\underline{\underline{K}}_{FF}\,\boldsymbol{w}_F \;=\; \boldsymbol{p}_F
\qquad \text{mit} \qquad \bar{p} = \frac{p_0}{\sigma_0\,t}.
$$

In [ ]:
# sigma_0 * t in den Nenner des Lastvektors ziehen
p_bar  = p0 / (sigma0 * t)
p_load = p_bar * p

# Reduziertes System loesen
free    = ~is_boundary
w       = np.zeros(len(nodes))
w[free] = np.linalg.solve(K[np.ix_(free, free)], p_load[free])

w_max = w.max()
print(f"Maximale Auslenkung  w_max = {w_max*1e3:.3f} mm")

## Vergleich mit analytischer Lösung

Für eine rechteckige, an allen Rändern eingespannte Membran liefert die Fourier-Reihe

$$
w_{\max} \;=\; \frac{16\,p_0}{\pi^4 \sigma_0 t}
\sum_{m,n \text{ ungerade}} \frac{(-1)^{(m+n-2)/2}}{m\,n\,(m^2/L_x^2 + n^2/L_y^2)} .
$$

Schon wenige Terme ergeben einen guten Vergleichswert.

In [ ]:
def w_max_analytic(p0, sigma0, t, Lx, Ly, terms=15):
    s = 0.0
    for m in range(1, 2*terms, 2):
        for n in range(1, 2*terms, 2):
            sign = (-1)**((m-1)//2 + (n-1)//2)
            s += sign / (m * n * (m**2/Lx**2 + n**2/Ly**2))
    return 16 * p0 / (np.pi**4 * sigma0 * t) * s


w_ref = w_max_analytic(p0, sigma0, t, Lx, Ly)
print(f"FEM         w_max = {w_max*1e3:.3f} mm")
print(f"Analytisch  w_max = {w_ref*1e3:.3f} mm")
print(f"Abweichung        = {(w_max - w_ref)/w_ref*100:+.2f} %")

## Visualisierung: 2D-Konturplot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_aspect("equal")
contour = ax.tricontourf(nodes[:,0], nodes[:,1], elements, w*1e3,
                         levels=15, cmap="plasma_r")
ax.triplot(nodes[:,0], nodes[:,1], elements, color="k", linewidth=0.2)
plt.colorbar(contour, ax=ax, label="Auslenkung w [mm]")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title(rf"$\sigma_0 = {sigma0/1e6:.0f}$ MPa, $t = {t*1e3:.1f}$ mm, "
             rf"$p_0 = {p0/1e3:.0f}$ kPa, $w_{{\max}} = {w_max*1e3:.2f}$ mm")
plt.show()

## Visualisierung: 3D-Fläche

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection="3d")
ax.set_box_aspect((Lx, Ly, 0.4*max(Lx, Ly)))
ax.view_init(elev=30, azim=-50)
ax.plot_trisurf(nodes[:,0], nodes[:,1], w*1e3, triangles=elements,
                cmap="plasma_r", edgecolor="k", linewidth=0.2, alpha=0.9)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("w [mm]")
ax.set_title(rf"Membran-Auslenkung (max = {w_max*1e3:.2f} mm)")
plt.show()

## Was, wenn?

Probiere Folgendes (Werte oben anpassen, dann alle Zellen erneut laufen lassen):

- **Höhere Vorspannung** $\sigma_0$  -- Wie ändert sich $w_{\max}$?
- **Dünnere Membran** $t$  -- Welchen Effekt hat das?
- **Feineres Netz** (`nx, ny` grösser)  -- Konvergiert die FEM-Lösung gegen den analytischen Wert?
- **Anderes Seitenverhältnis** $L_x \neq L_y$  -- Wo liegt das Maximum dann?

**Theorie-Skalierung:** Aus der DGL folgt $w \propto p_0 / (\sigma_0 t)$. Verdoppelt man
$\sigma_0$, halbiert sich also $w_{\max}$.

## Alternative Aufgabe: E-förmige Membran

Bisher hatten wir es einfach: das rechteckige Gebiet lässt sich mit einem `np.meshgrid` direkt
vernetzen. Für **beliebige Polygon-Geometrien** brauchen wir mehr Aufwand:

1. **Randknoten** entlang jeder Polygon-Kante gleichmässig verteilen.
2. **Innenknoten** als regelmässiges Gitter im umschliessenden Rechteck, dann ausserhalb
   liegende Knoten verwerfen.
3. **Delaunay-Triangulation** über alle Knoten, anschliessend Dreiecke verwerfen, deren
   Schwerpunkt ausserhalb des Polygons liegt.

Die Zelle unten implementiert dieses Schema für eine Membran in Form des Buchstabens **E**.
Sie definiert dieselben Variablen wie der Rechteck-Block oben (`nodes`, `elements`,
`is_boundary`, `Lx`, `Ly`, `t`, `sigma0`, `p0`).

**Verwendung als alternatives Problem:**
Diesen Code-Block in die Zelle [_Membran-Geometrie und Parameter_](#Modelldefinition-und-Vernetzung)
ganz oben einsetzen (oder die Zelle hier ausführen und ab der Assemblierung neu durchlaufen).
Die analytische Vergleichszelle gilt nur für Rechtecke und sollte für die E-Membran übersprungen werden.

In [ ]:
# === Alternative: E-foermige Membran mit Polygon-Vernetzung ==================
from scipy.spatial   import Delaunay
from matplotlib.path import Path

# Polygon-Eckpunkte (Buchstabe E), gegen Uhrzeigersinn
vertices = np.array([
    [0, 0], [3, 0], [3, 1  ], [1, 1  ], [1, 1.5], [2, 1.5],
    [2, 2.5], [1, 2.5], [1, 3  ], [3, 3  ], [3, 4  ], [0, 4  ],
], dtype=float)

# Membran-Parameter (wie bei der Rechteck-Membran)
t      = 0.5e-3
sigma0 = 50e6
p0     = 5e3
h_mesh = 0.15        # ungefaehre Dreiecks-Kantenlaenge [m]


def mesh_polygon(vertices, h):
    """Vernetzt ein Polygon mit Dreieckselementen.
    Liefert nodes (N, 2), elements (M, 3) und is_boundary (N,)."""

    # 1) Randknoten gleichmaessig auf jeder Polygon-Kante
    boundary = []
    for k, p in enumerate(vertices):
        q = vertices[(k + 1) % len(vertices)]
        n = max(int(np.linalg.norm(q - p) / h), 1)
        boundary.append(np.linspace(p, q, n, endpoint=False))
    boundary = np.vstack(boundary)

    # 2) Innenknoten: Gitter in Bounding Box, ausserhalb-Polygon verwerfen
    xmin, ymin = vertices.min(axis=0)
    xmax, ymax = vertices.max(axis=0)
    xg = np.linspace(xmin, xmax, int((xmax - xmin) / h) + 1)
    yg = np.linspace(ymin, ymax, int((ymax - ymin) / h) + 1)
    Xg, Yg   = np.meshgrid(xg, yg)
    interior = np.column_stack((Xg.ravel(), Yg.ravel()))
    poly     = Path(vertices)
    interior = interior[poly.contains_points(interior, radius=-0.4*h)]

    # 3) Triangulieren, Dreiecke mit Schwerpunkt ausserhalb verwerfen
    nodes     = np.vstack((boundary, interior))
    elements  = Delaunay(nodes).simplices
    centroids = nodes[elements].mean(axis=1)
    elements  = elements[poly.contains_points(centroids)]

    # Randknoten = die ersten len(boundary) Eintraege
    is_boundary = np.arange(len(nodes)) < len(boundary)
    return nodes, elements, is_boundary


nodes, elements, is_boundary = mesh_polygon(vertices, h_mesh)

# Bounding-Box-Abmessungen fuer 3D-Plot-Skalierung
Lx = vertices[:, 0].max() - vertices[:, 0].min()
Ly = vertices[:, 1].max() - vertices[:, 1].min()

print(f"Knoten:   {len(nodes):4d}  ({is_boundary.sum()} davon am Rand)")
print(f"Elemente: {len(elements):4d}")

# Netz visualisieren
fig, ax = plt.subplots(figsize=(7, 9))
ax.set_aspect("equal")
ax.triplot(nodes[:, 0], nodes[:, 1], elements, color="k", linewidth=0.4)
ax.plot(nodes[~is_boundary, 0], nodes[~is_boundary, 1], "ko",
        markersize=2.5, label="Innenknoten")
ax.plot(nodes[ is_boundary, 0], nodes[ is_boundary, 1], "go",
        markersize=4.5, label="Randknoten (w=0)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title(f"E-foermige Membran  ({len(elements)} Elemente)")
ax.legend()
plt.show()